# Classical ML Advanced — SVMs, Gaussian Processes, XGBoost, UMAP
## The tools you need for when deep learning is overkill

**TL;DR:** Deep learning is not always the answer. For small datasets (< 10k samples), structured/tabular data, or when you need interpretability, classical ML wins. SVMs give you the maximum-margin geometric intuition. Gaussian Processes give you uncertainty estimates for free. XGBoost wins most tabular ML competitions. UMAP reduces your 1024-dim protein embeddings to 2D for visualization. This notebook covers all four deeply — with theory, implementation, and bioinformatics applications.

## Beginner Teaching Frame

**Who should fully work through this notebook:** students with little or no ML background.

**How to study it on a first pass:**
- read every markdown section before touching the code
- run the code in small chunks
- paraphrase each concept in your own words
- write one tiny example from memory after finishing

**Common traps:** memorizing syntax without understanding the data flow, skipping probability, and rushing past train/test split discipline.


## Canonical Resource Upgrade

If you need a second explanation, use these first:
- [CS50P](https://cs50.harvard.edu/python/) for programming fundamentals
- [Harvard Stat 110](https://stat110.hsites.harvard.edu/) for probability intuition
- [scikit-learn MOOC](https://inria.github.io/scikit-learn-mooc/) for correct ML workflow
- [PyTorch Tutorials](https://docs.pytorch.org/tutorials/) once you reach the deep learning notebooks


## 🗺️ Prerequisites & Learning Path
| | |
|--|--|
| 🔑 Prerequisites | 00/02 ML Fundamentals, 00/08 Math Foundations (Lagrangians, kernels) |
| 📍 You Are Here | Module 00/09 — Classical ML Advanced |
| ➡️ Next Steps | 04/01 ML for Omics (apply these to genomics) |
| 🏁 Goal | Implement SVM kernel trick, train XGBoost, use GP for protein engineering, visualize with UMAP |

### 🆕 From Scratch? Start Here:
1. [StatQuest SVM](https://www.youtube.com/watch?v=efR1C6CvhmE) — visual, 30 min
2. [StatQuest XGBoost](https://www.youtube.com/watch?v=OtD8wVaFm6E) — visual, 25 min
3. 00/02 ML Fundamentals notebook
4. This notebook

**Time:** 12–18 hours | **Difficulty:** ⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: 00/02 (sklearn basics), 00/08 (math: kernels, Lagrangians)
- Used by: 04/01 (gene expression — SVMs outperform DL at small N), 10/01 (GP for protein engineering)

## PART 1 — Support Vector Machines

SVMs find the **maximum-margin hyperplane** separating two classes. The key insight is that only the training points closest to the decision boundary (support vectors) matter — the rest can be removed.

The **kernel trick** lets SVMs work in infinite-dimensional feature spaces without ever computing the transformation explicitly — because the dual formulation only needs dot products between data points.

In [ ]:
# Support Vector Machines with Kernels
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_moons, make_classification

np.random.seed(42)

# SVM with RBF kernel for nonlinear classification
X, y = make_moons(n_samples=200, noise=0.2, random_state=42)
sc = StandardScaler()
X_s = sc.fit_transform(X)

kernels = ['linear', 'rbf', 'poly']
print("SVM kernel comparison (5-fold CV accuracy):")
for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, gamma='scale')
    scores = cross_val_score(svm, X_s, y, cv=5)
    print(f"  {kernel:8s}: {scores.mean():.3f} ± {scores.std():.3f}")

# K-mer SVM for DNA classification (spectrum kernel concept)
def spectrum_kernel(s1, s2, k=3):
    """Spectrum kernel: dot product in k-mer feature space."""
    from collections import Counter
    v1 = Counter(s1[i:i+k] for i in range(len(s1)-k+1))
    v2 = Counter(s2[i:i+k] for i in range(len(s2)-k+1))
    common = set(v1) & set(v2)
    return sum(v1[kmer] * v2[kmer] for kmer in common)

seqs = ["ATGCGATCG", "ATGCGATCG", "TTTTCCCCGG", "GCGCGCGCGC"]
print("\nSpectrum kernel (k=3) between sequences:")
for i in range(len(seqs)):
    for j in range(i, len(seqs)):
        k = spectrum_kernel(seqs[i], seqs[j])
        print(f"  K(seq{i}, seq{j}) = {k}")

> **Expected output:** Spectrum kernel values between DNA sequences, e.g., `K(seq0, seq0) = 7`, `K(seq0, seq1) = 7`  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

## PART 2 — Gaussian Processes

A **Gaussian Process** is a distribution over functions. Rather than learning a single function, GP inference gives you a posterior distribution over all functions consistent with the data — including uncertainty estimates at every point.

This makes GPs uniquely powerful for **Bayesian optimization** in protein engineering: use the GP's uncertainty to decide which experiments to run next.

In [ ]:
# Gaussian Processes: Bayesian Non-parametric Regression
import numpy as np
from scipy.spatial.distance import cdist

def rbf_kernel(X1, X2, length_scale=1.0, sigma_f=1.0):
    """RBF (Squared Exponential) kernel."""
    dist_sq = cdist(X1.reshape(-1, 1), X2.reshape(-1, 1), 'sqeuclidean')
    return sigma_f**2 * np.exp(-0.5 * dist_sq / length_scale**2)

def gp_posterior(X_train, y_train, X_test, length_scale=1.0, noise=0.1):
    """GP posterior mean and variance."""
    K_tt = rbf_kernel(X_train, X_train, length_scale) + noise**2 * np.eye(len(X_train))
    K_ts = rbf_kernel(X_train, X_test, length_scale)
    K_ss = rbf_kernel(X_test, X_test, length_scale)

    K_inv = np.linalg.inv(K_tt)
    mu = K_ts.T @ K_inv @ y_train
    cov = K_ss - K_ts.T @ K_inv @ K_ts
    return mu, np.diag(cov)

# Protein binding affinity prediction with uncertainty
np.random.seed(42)
X_train = np.array([-3, -1, 0.5, 2, 3.5], dtype=float)
y_train = np.sin(X_train) + 0.2 * np.random.randn(len(X_train))
X_test = np.linspace(-4, 4, 20)

mu, var = gp_posterior(X_train, y_train, X_test, length_scale=1.0, noise=0.1)
std = np.sqrt(np.maximum(var, 0))

print("GP Posterior (protein binding affinity):")
print(f"{'x':>6} {'μ':>8} {'σ':>8}")
for x, m, s in zip(X_test[::4], mu[::4], std[::4]):
    print(f"  {x:6.2f} {m:8.4f} {s:8.4f}")
print(f"\nHigh uncertainty at x=±4 (far from training data)")
print(f"Low uncertainty near training points {X_train}")
print("GP: provides uncertainty estimates — crucial for drug discovery")

## PART 3 — XGBoost

**Gradient boosting** builds an ensemble of trees sequentially, where each new tree fits the negative gradient (residuals) of the previous ensemble. XGBoost adds regularization, second-order gradients, and column subsampling — making it the go-to for structured/tabular data.

Rule of thumb: **for tabular bioinformatics data with < 100k rows, try XGBoost before any deep learning.**

In [ ]:
# XGBoost: Gradient Boosting for Genomics
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score

np.random.seed(42)
# Simulate: gene expression features for cancer subtype prediction
X, y = make_classification(n_samples=300, n_features=50, n_informative=15,
                            n_classes=2, random_state=42)

# GradientBoosting (scikit-learn, similar to XGBoost)
gb_params = [
    {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 4},
    {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 5},
]

print("Gradient Boosting hyperparameter comparison:")
best_params, best_score = None, 0
for params in gb_params:
    gb = GradientBoostingClassifier(**params, random_state=42)
    scores = cross_val_score(gb, X, y, cv=5, scoring='roc_auc')
    avg = scores.mean()
    print(f"  n_est={params['n_estimators']}, lr={params['learning_rate']}, depth={params['max_depth']}: ROC-AUC={avg:.3f}")
    if avg > best_score:
        best_score, best_params = avg, params

print(f"\nBest: {best_params} → {best_score:.3f}")

# Feature importance
best_gb = GradientBoostingClassifier(**best_params, random_state=42).fit(X, y)
top5 = np.argsort(best_gb.feature_importances_)[-5:][::-1]
print("\nTop 5 features (gene importances):")
for rank, fi in enumerate(top5, 1):
    print(f"  #{rank}: feature_{fi} → importance={best_gb.feature_importances_[fi]:.4f}")

## PART 4 — UMAP for Protein Embeddings

**UMAP** (Uniform Manifold Approximation and Projection) is the best general-purpose tool for visualizing high-dimensional biological data. ESM2 gives you 1280-dim protein embeddings — UMAP reduces these to 2D while preserving both local clusters and global relationships better than t-SNE.

Key use case: visualize 10,000 protein embeddings colored by family, function, or evolutionary origin.

In [ ]:
# Classical ML Advanced: Summary
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

np.random.seed(42)
X = np.random.randn(200, 30)
y = (X[:, 0]**2 + X[:, 1] > 1).astype(int)

# Pipeline with PCA + SVM
pipe = Pipeline([
    ('sc', StandardScaler()),
    ('pca', PCA(n_components=10)),
    ('svm', SVC(kernel='rbf', C=1.0))
])
scores = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')
print(f"PCA(10) + SVM: ROC-AUC = {scores.mean():.3f}")

# Gradient boosting
gb = Pipeline([('sc', StandardScaler()), ('gb', GradientBoostingClassifier(n_estimators=50))])
scores = cross_val_score(gb, X, y, cv=5, scoring='roc_auc')
print(f"GradientBoosting: ROC-AUC = {scores.mean():.3f}")

print("\nKey insight: SVM+kernel=90s method, GBM=2000s, DNN=2010s+")
print("For tabular omics: GBM often beats DNN for small/medium datasets")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **SVM:** Vapnik 1995 (original SVM), Scholkopf & Smola "Learning with Kernels" — kernel methods theory
- **GP:** Rasmussen & Williams "Gaussian Processes for ML" — free PDF, definitive reference
- **XGBoost:** Chen & Guestrin 2016 (XGBoost paper), Friedman 2001 (gradient boosting)
- **UMAP:** McInnes 2018 (UMAP paper) — Riemannian manifold approximation

### 2️⃣ Must-Have Popular Resources
- 📘 **Book:** [Gaussian Processes for ML (Rasmussen)](http://www.gaussianprocess.org/gpml/) — free PDF, THE GP book
- 📘 **Book:** Elements of Statistical Learning ch. 12 (SVMs) + ch. 10 (boosting) — free PDF
- 🎓 **Video:** StatQuest SVMs, XGBoost, Random Forests (Josh Starmer, YouTube) — best visual explanations
- ⭐ **GitHub:** [dmlc/xgboost](https://github.com/dmlc/xgboost) — 26k★ the original library
- ⭐ **GitHub:** [lmcinnes/umap](https://github.com/lmcinnes/umap) — 7k★ UMAP implementation
- 🤗 **HuggingFace:** protein embedding datasets for UMAP visualization
- 📊 **Kaggle:** [Tabular Playground competitions](https://www.kaggle.com/competitions?listOption=completed&categoryIds=1) — XGBoost dominates

### 3️⃣ Practicals — Hands-On
- 💻 Implement a GP from scratch, verify posterior equals scipy.stats output
- 💻 Run Bayesian optimization with GP on a protein engineering benchmark (ProteinGym)
- 💻 Compare RF vs XGBoost vs SVM on gene expression cancer classification
- 📊 Kaggle: [Any tabular competition](https://www.kaggle.com/competitions) — XGBoost is the baseline
- 🤗 HuggingFace: UMAP-visualize ESM2 embeddings for 1000 proteins from UniProt

### 4️⃣ Real-World Problems
- 🏭 Every drug discovery company uses GP + Bayesian optimization for wet lab experimental design
- 🏭 ClinVar pathogenicity scoring uses gradient boosting features
- 📊 **Dataset:** [ProteinGym](https://huggingface.co/datasets/OATML-Markslab/ProteinGym) — fitness landscapes for GP-BO
- 🔬 **Application:** Bayesian optimization for directed evolution — maximize enzyme thermostability

### 5️⃣ Interview Tips
- ❓ **Must know:** Why does the kernel trick work? (dual form only depends on dot products)
- ❓ **Must know:** What are support vectors? (data points on the margin that define the hyperplane)
- ❓ **Must know:** GP posterior formula — derive from Bayes rule for Gaussians
- ⚠️ **Common mistake:** Using UMAP without checking n_neighbors and min_dist — default settings may not be right
- 💡 **Pro tip:** For drug discovery interviews, know GP-BO cold — it is the standard for experimental design

### 6️⃣ Hot Industry Topics
- 🔥 **Trending:** [BoTorch](https://github.com/pytorch/botorch) — PyTorch Bayesian optimization library (Meta)
- 🔥 **Trending:** [LightGBM](https://github.com/microsoft/LightGBM) — Microsoft's faster XGBoost alternative
- 🔥 **Trending:** [SHAP](https://github.com/shap/shap) — 22k★ model interpretability via Shapley values
- 🚀 **Build:** GP-Bayesian optimization loop for protein fitness — pick 10 mutations to test, update GP, repeat
- 🚀 **Build:** UMAP visualization of ESM2 embeddings for 5 protein families — identify outliers

## Interview Q&A — Classical ML Advanced

**Q: What is the kernel trick and why does it allow SVMs to work in infinite-dimensional spaces?**
A: The kernel trick: instead of explicitly computing φ(x) (high/infinite-dim feature map), compute K(x,y) = φ(x)·φ(y) directly. The SVM dual only needs dot products, never φ(x) explicitly. The RBF kernel K(x,y) = exp(-||x-y||²/2σ²) corresponds to an infinite-dimensional feature space, yet is computed in O(d) time. This is the "trick" — work in infinite dimensions at O(d) cost.

**Q: Why are Gaussian Processes preferred over neural networks for small datasets?**
A: GPs give calibrated uncertainty: posterior predictive is a distribution, not a point. With 10-50 labeled protein variants, a GP with a good kernel (e.g., Matérn) outperforms a neural network that would overfit. GPs also enable active learning: query the next experiment where uncertainty is highest (Expected Improvement). Neural networks need hundreds of examples to generalize.

**Q: What is the difference between Random Forest and Gradient Boosting (XGBoost)?**
A: Random Forest: builds trees in parallel, averages them (bagging), each tree uses a random feature subset. High variance of individual trees gets averaged out. Gradient Boosting: builds trees sequentially, each correcting the residuals of the previous. Fits the gradient of the loss. XGBoost adds: regularization, handling missing values, column subsampling. For tabular protein features: XGBoost usually wins; RF is faster and more robust to hyperparameters.

**Q: When would you use UMAP over t-SNE?**
A: UMAP for: large datasets (>10K), scalable (O(n log n) vs t-SNE's O(n²)), preserves global structure better, can be used for new data projection (fit on train, transform test). t-SNE for: publication-quality 2D visualizations with 1K-10K points, when local cluster separation matters most. For protein embeddings from ESM2: UMAP is standard.

**Q: How does XGBoost handle missing values automatically?**
A: XGBoost learns a default direction for missing values at each split during training: it tries both left and right branches for missing values and picks the one that reduces loss more. At inference, missing values follow the learned default path. For protein data with variable-length features or missing annotations: this matters.

## Mastery Check

Before moving on, make sure you can:
1. explain the core concept of this notebook in plain English without looking
2. write a small toy example from scratch
3. identify one common mistake and explain why it is wrong
4. say whether you should revisit math, Python, or ML basics before continuing


---
## 📚 Resources — Classical ML & Advanced scikit-learn

### University Courses (All Free)
| Course | Best For |
|--------|----------|
| **[Stanford CS229 (Andrew Ng)](https://cs229.stanford.edu/)** | Rigorous math: SVMs, trees, EM algorithm. Lecture notes PDF free. |
| **[MIT 6.036 Intro ML](https://introml.mit.edu/)** | Best for building intuition; free problem sets with solutions |
| **[Harvard CS181 ML](https://harvard-ml-courses.github.io/cs181-web/)** | Detailed treatment of regularization, kernels, ensembles. Notes free. |
| **[CalTech Learning from Data](https://work.caltech.edu/telecourse)** | Best theoretical foundation; Prof. Abu-Mostafa; free video |

### Essential Reading (Free PDFs)
- **[The Elements of Statistical Learning](https://hastie.su.domains/ElemStatLearn/)** (Hastie, Tibshirani, Friedman) — the graduate-level reference. Free PDF from Stanford.
- **[Pattern Recognition and Machine Learning](https://www.microsoft.com/en-us/research/uploads/prod/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf)** (Bishop) — Bayesian perspective. Free PDF.
- **[Understanding Machine Learning](https://www.cs.huji.ac.il/~shais/UnderstandingMachineLearning/)** (Shalev-Shwartz) — PAC learning theory. Free PDF from Hebrew U.

### What Interviewers Actually Test on Classical ML
1. **Random Forests**: why bagging reduces variance but not bias; why feature randomness helps
2. **Gradient Boosting**: difference from bagging; why XGBoost is fast; when to use vs deep learning
3. **SVM**: kernel trick; support vectors; why L2 regularization is implicit in SVM
4. **Cross-validation**: why stratified; why GroupKFold for protein data
5. **Feature importance**: SHAP vs permutation importance vs impurity-based — which to trust and when

### Scikit-learn Pipeline (The Correct Pattern)
```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold

# ALWAYS use Pipeline to prevent data leakage
pipe = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier())])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
# The scaler is fit ONLY on training folds — this is correct
```

### MIT Problem Sets (Practice)
- **[MIT 6.036 Homework 6: Kernel Methods](https://introml.mit.edu/)** — implement kernel SVM from scratch
- **[Stanford CS229 Problem Set 2](https://cs229.stanford.edu/past_exams.html)** — bias-variance, regularization problems


---
## 🐛 Debug Exercise — Classical ML Pipeline Bugs

Find and fix **3 bugs** in this ML evaluation pipeline.

<details><summary>Show bugs</summary>

**Bug 1:** Scaler is fit on the full dataset before splitting — classic data leakage. Must fit only on X_train.

**Bug 2:** GridSearchCV `refit=False` means it won't refit the best model — `best_estimator_` won't be available. Should be `refit=True` (default).

**Bug 3:** `roc_auc_score` called with raw predictions instead of probabilities. Need `predict_proba(X_test)[:,1]`, not `predict(X_test)`.

</details>

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
np.random.seed(42)

X, y = make_classification(n_samples=200, n_features=10, random_state=42)

# BUG 1: scaler fit before split -> data leakage
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)       # should fit ONLY on X_train
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# BUG 2: refit=False prevents access to best_estimator_
param_grid = {"C": [0.1, 1, 10]}
gs = GridSearchCV(SVC(probability=True), param_grid, cv=3, refit=False)  # should be refit=True
gs.fit(X_train, y_train)

try:
    best = gs.best_estimator_
    preds = best.predict(X_test)
    # BUG 3: predict() returns class labels, not probabilities
    auc = roc_auc_score(y_test, preds)   # should use predict_proba(X_test)[:,1]
    print(f"AUC: {auc:.4f}")
except Exception as e:
    print(f"Error: {e}")
    print("Fix Bug 2 first (refit=False prevents best_estimator_)")


## Notebook Complete

**You can now:**
- Implement and tune SVMs, gradient boosting, and ensemble methods
- Apply feature engineering and preprocessing pipelines with scikit-learn

**Where this fits:**
- Previous: 08_mathematical_foundations
- Next: 10_how_to_read_a_paper — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes